# Neural Networks for Spend Category Prediction

We will:
- Scale features for stable neural network training
- Build a baseline fully-connected neural network
- Train improved models with:
  - More layers / neurons
  - Regularization (Dropout, L2)
  - Learning rate scheduling or early stopping


### Import Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report

import os
import joblib
import warnings
warnings.filterwarnings("ignore")

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers,regularizers

2025-11-26 16:23:33.190894: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-26 16:23:33.248679: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-26 16:23:34.348066: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


### Load datasets + Standard Scaling

In [2]:
X_train = pd.read_csv("../data/processed/X_train.csv")
X_val   = pd.read_csv("../data/processed/X_val.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_val   = pd.read_csv("../data/processed/y_val.csv").squeeze()

test = pd.read_csv("../data/processed/test.csv")
test_trip_id = pd.read_csv("../data/processed/test_trip_id.csv").squeeze()

print("Train shape:", X_train.shape)
print("Val shape:", X_val.shape)
print("Test shape:", test.shape)

# Standard Scaling for Neural Networks

scaler = StandardScaler()
scaler.fit(X_train)  # only train

X_train_scaled = scaler.transform(X_train)
X_val_scaled   = scaler.transform(X_val)
test_scaled    = scaler.transform(test)

print("Scaling done.")

joblib.dump(scaler, "../models/nn_scaler.joblib")
print("Scaler saved -> ../models/nn_scaler.joblib")


Train shape: (10096, 79)
Val shape: (2524, 79)
Test shape: (5852, 79)
Scaling done.
Scaler saved -> ../models/nn_scaler.joblib


### Baseline Fully Connected Neural Network

In [3]:
# Convert labels to one-hot (required for NN)
num_classes = 3

y_train_oh = keras.utils.to_categorical(y_train, num_classes=num_classes)
y_val_oh   = keras.utils.to_categorical(y_val, num_classes=num_classes)

model_base = keras.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(num_classes, activation="softmax")
])

model_base.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model_base.summary()

# Train Baseline Model
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history_base = model_base.fit(
    X_train_scaled,
    y_train_oh,
    validation_data=(X_val_scaled, y_val_oh),
    epochs=40,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

val_probs = model_base.predict(X_val_scaled)
val_preds = np.argmax(val_probs, axis=1)

acc = accuracy_score(y_val, val_preds)
f1  = f1_score(y_val, val_preds, average="macro")

print("\nBaseline Neural Network Performance")
print("--------------------------------------")
print(f"Accuracy: {acc:.4f}")
print(f"F1 Macro: {f1:.4f}\n")

print("Classification Report:")
print(classification_report(y_val, val_preds))

model_base.save("../models/nn_baseline.keras")
print("Saved -> ../models/nn_baseline.keras")


test_probs = model_base.predict(test_scaled)
test_preds = np.argmax(test_probs, axis=1)

submission_nn = pd.DataFrame({
    "trip_id": test_trip_id,
    "spend_category": test_preds
})

submission_nn.to_csv("../submissions/nn_baseline.csv", index=False)

print("Submission created -> ../submissions/nn_baseline.csv")

2025-11-26 16:23:35.065621: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         5,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,299 (28.51 KB)

 Trainable params: 7,299 (28.51 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/40
158/158 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6833 - loss: 0.7486 - val_accuracy: 0.7183 - val_loss: 0.6514
Epoch 2/40
158/158 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7319 - loss: 0.6371 - val_accuracy: 0.7286 - val_loss: 0.6351
Epoch 3/40
158/158 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7401 - loss: 0.6107 - val_accuracy: 0.7397 - val_loss: 0.6233
Epoch 4/40
158/158 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7514 - loss: 0.5938 - val_accuracy: 0.7417 - val_loss: 0.6250
Epoch 5/40
158/158 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7564 - loss: 0.5784 - val_accuracy: 0.7409 - val_loss: 0.6267
Epoch 6/40
158/158 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7636 - loss: 0.5674 - val_accuracy: 0.7409 - val_loss: 0.6282
Epoch 7/40
158/158 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7712 - loss: 0.5541 - val_accuracy: 0.7345 - val_loss: 0.6401
Epoch 8/40
158/158 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7751 - loss: 0.5444 - val_accuracy: 0.

### Class weighted and Regularization

In [32]:
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras import backend as K
classes = np.unique(y_train)
raw_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)
class_weights = dict(zip(classes, raw_weights))

print("\nComputed Class Weights:")
print(class_weights)

model_weighted = keras.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),
    layers.Dense(512, activation="swish",
                 kernel_regularizer=regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.Dropout(0.25),
    
    layers.Dense(256, activation="swish",
                 kernel_regularizer=regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.Dropout(0.25),
    
    layers.Dense(128, activation="swish",
                 kernel_regularizer=regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.Dropout(0.25),
    
    layers.Dense(64, activation="swish",
                 kernel_regularizer=regularizers.l2(1e-4)),
    layers.BatchNormalization(),

    layers.Dense(num_classes, activation="softmax")
])

model_weighted.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model_weighted.summary()


early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-5,
    verbose=1
)

print("\nTraining model WITH class weights...\n")

history_weighted = model_weighted.fit(
    X_train_scaled,
    y_train_oh,
    validation_data=(X_val_scaled, y_val_oh),
    epochs=40,
    batch_size=128,
    callbacks=[early_stop, reduce_lr],
    class_weight=class_weights,
    verbose=1
)


val_probs_w = model_weighted.predict(X_val_scaled)
val_preds_w = np.argmax(val_probs_w, axis=1)

acc_w = accuracy_score(y_val, val_preds_w)
f1_w  = f1_score(y_val, val_preds_w, average="macro")

print("\nCLASS WEIGHTED MODEL PERFORMANCE")
print("--------------------------------------")
print(f"Accuracy: {acc_w:.4f}")
print(f"Macro F1: {f1_w:.4f}\n")
print(classification_report(y_val, val_preds_w))


model_weighted.save("../models/nn_class_weighted.keras")
print("Saved -> ../models/nn_class_weighted.keras")

# Predict on test
test_probs_w = model_weighted.predict(test_scaled)
test_preds_w = np.argmax(test_probs_w, axis=1)

submission_nn_w = pd.DataFrame({
    "trip_id": test_trip_id,
    "spend_category": test_preds_w
})

submission_nn_w.to_csv("../submissions/nn_class_weighted_tuned.csv", index=False)

print("Submission created -> ../submissions/nn_class_weighted_tuned.csv")



Computed Class Weights:
{np.float64(0.0): np.float64(0.6736055511075527), np.float64(1.0): np.float64(0.8565368626452872), np.float64(2.0): np.float64(2.873896954170225)}


Model: "sequential_28"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_123 (Dense)               │ (None, 512)            │        40,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_93          │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_66 (Dropout)            │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_124 (Dense)               │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_94          │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_67 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_125 (Dense)               │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_95          │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_68 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_126 (Dense)               │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_96          │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_127 (Dense)               │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 217,475 (849.51 KB)

 Trainable params: 215,555 (842.01 KB)

 Non-trainable params: 1,920 (7.50 KB)


Training model WITH class weights...

Epoch 1/40
79/79 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.6155 - loss: 0.9639 - val_accuracy: 0.7139 - val_loss: 0.8361 - learning_rate: 0.0010
Epoch 2/40
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.6677 - loss: 0.8307 - val_accuracy: 0.6933 - val_loss: 0.7724 - learning_rate: 0.0010
Epoch 3/40
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.6767 - loss: 0.7994 - val_accuracy: 0.6830 - val_loss: 0.7658 - learning_rate: 0.0010
Epoch 4/40
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.6765 - loss: 0.7827 - val_accuracy: 0.6866 - val_loss: 0.7631 - learning_rate: 0.0010
Epoch 5/40
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.6918 - loss: 0.7672 - val_accuracy: 0.6929 - val_loss: 0.7602 - learning_rate: 0.0010
Epoch 6/40
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.6951 - loss: 0.7553 - val_accuracy: 0.6846 - val_loss: 0.7842 - learning_rate: 0.0010
Epoch 7/40
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy

### 

In [5]:
from tensorflow.keras import backend as K


def focal_loss(gamma=2., alpha=None):
    def loss_fn(y_true, y_pred):
        y_pred = K.clip(y_pred, K.epsilon(), 1. - K.epsilon())
        ce = -y_true * K.log(y_pred)
        weight = K.pow(1 - y_pred, gamma)

        if alpha is not None:
            alpha_tensor = tf.convert_to_tensor(alpha, dtype=tf.float32)
            alpha_weight = y_true * alpha_tensor
        else:
            alpha_weight = 1

        return K.sum(alpha_weight * weight * ce, axis=1)
    return loss_fn

classes = np.unique(y_train)
raw_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)
alpha = raw_weights / raw_weights.sum() 
alpha

model_focal = keras.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),
    
    layers.Dense(256, activation="relu",
                 kernel_regularizer=regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.Dropout(0.25),
    
    layers.Dense(128, activation="relu",
                 kernel_regularizer=regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.Dropout(0.25),
    
    layers.Dense(64, activation="relu",
                 kernel_regularizer=regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    
    layers.Dense(num_classes, activation="softmax")
])

model_focal.compile(
    optimizer=keras.optimizers.Adam(0.001),
    loss=focal_loss(gamma=2, alpha=alpha),
    metrics=["accuracy"]
)

model_focal.summary()


early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=6,
    restore_best_weights=True
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    verbose=1
)

history_focal = model_focal.fit(
    X_train_scaled,
    y_train_oh,
    validation_data=(X_val_scaled, y_val_oh),
    epochs=60,
    batch_size=64,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

val_probs_f = model_focal.predict(X_val_scaled)
val_preds_f = np.argmax(val_probs_f, axis=1)

acc_f = accuracy_score(y_val, val_preds_f)
f1_f  = f1_score(y_val, val_preds_f, average="macro")

print("\nFocal Loss Model Performance")
print("--------------------------------")
print(f"Accuracy: {acc_f:.4f}")
print(f"Macro F1: {f1_f:.4f}\n")
print(classification_report(y_val, val_preds_f))

model_focal.save("../models/nn_focal.keras")
print("ASaved -> ../models/nn_focal.keras")

test_probs_f = model_focal.predict(test_scaled)
test_preds_f = np.argmax(test_probs_f, axis=1)

submission_focal = pd.DataFrame({
    "trip_id": test_trip_id,
    "spend_category": test_preds_f
})

submission_focal.to_csv("../submissions/nn_focal.csv", index=False)
print("Submission created -> ../submissions/nn_focal.csv")


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_7 (Dense)                 │ (None, 256)            │        20,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 63,619 (248.51 KB)

 Trainable params: 62,723 (245.01 KB)

 Non-trainable params: 896 (3.50 KB)

Epoch 1/60
158/158 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.5749 - loss: 0.1545 - val_accuracy: 0.6672 - val_loss: 0.1073 - learning_rate: 0.0010
Epoch 2/60
158/158 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6384 - loss: 0.1143 - val_accuracy: 0.6779 - val_loss: 0.1033 - learning_rate: 0.0010
Epoch 3/60
158/158 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6554 - loss: 0.1044 - val_accuracy: 0.6961 - val_loss: 0.1004 - learning_rate: 0.0010
Epoch 4/60
158/158 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6657 - loss: 0.0991 - val_accuracy: 0.7116 - val_loss: 0.0988 - learning_rate: 0.0010
Epoch 5/60
158/158 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6741 - loss: 0.0948 - val_accuracy: 0.7036 - val_loss: 0.0976 - learning_rate: 0.0010
Epoch 6/60
158/158 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6801 - loss: 0.0918 - val_accuracy: 0.7262 - val_loss: 0.0970 - learning_rate: 0.0010
Epoch 7/60
158/158 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6887 - loss: 0.0893 - 

In [9]:
from tensorflow.keras import backend as K
from sklearn.utils.class_weight import compute_class_weight

def focal_loss(gamma=2., alpha=None):
    def loss_fn(y_true, y_pred):
        y_pred = K.clip(y_pred, K.epsilon(), 1. - K.epsilon())
        ce = -y_true * K.log(y_pred)

        focal_weight = K.pow(1 - y_pred, gamma)

        if alpha is not None:
            alpha_tensor = tf.convert_to_tensor(alpha, dtype=tf.float32)
            alpha_weight = y_true * alpha_tensor
        else:
            alpha_weight = 1

        return K.sum(alpha_weight * focal_weight * ce, axis=1)
    return loss_fn


classes = np.unique(y_train)
raw_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)
alpha = raw_weights
print("\nAlpha weights:", alpha)

model_hybrid = keras.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),
    
    layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.Dropout(0.25),
    
    layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.Dropout(0.25),

    layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.Dropout(0.25),
    
    layers.Dense(32, activation="relu", kernel_regularizer=regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    
    layers.Dense(num_classes, activation="softmax")
])

model_hybrid.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss=focal_loss(gamma=2.5, alpha=alpha),
    metrics=["accuracy"]
)

print("\nStage 1: Training with focal loss...\n")

history_focal = model_hybrid.fit(
    X_train_scaled,
    y_train_oh,
    validation_data=(X_val_scaled, y_val_oh),
    epochs=25,
    batch_size=64,
    callbacks=[
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=3,
            verbose=1
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=6,
            restore_best_weights=True
        )
    ],
    verbose=1
)

print("\nStage 2: Fine-tuning with cross-entropy\n")
model_hybrid.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_ce = model_hybrid.fit(
    X_train_scaled,
    y_train_oh,
    validation_data=(X_val_scaled, y_val_oh),
    epochs=15,
    batch_size=64,
    verbose=1
)


val_probs_h = model_hybrid.predict(X_val_scaled)
val_preds_h = np.argmax(val_probs_h, axis=1)

acc_h = accuracy_score(y_val, val_preds_h)
f1_h  = f1_score(y_val, val_preds_h, average="macro")

print("\nHYBRID MODEL PERFORMANCE")
print("--------------------------------------")
print(f"Accuracy: {acc_h:.4f}")
print(f"Macro F1: {f1_h:.4f}")
print("\nClassification Report:")
print(classification_report(y_val, val_preds_h))


model_hybrid.save("../models/nn_hybrid_focal_ce.keras")
print(" Saved -> ../models/nn_hybrid_focal_ce.keras")

test_probs_h = model_hybrid.predict(test_scaled)
test_preds_h = np.argmax(test_probs_h, axis=1)

submission_h = pd.DataFrame({
    "trip_id": test_trip_id,
    "spend_category": test_preds_h
})

submission_h.to_csv("../submissions/nn_hybrid_focal_ce.csv", index=False)

print("Submission created -> ../submissions/nn_hybrid_focal_ce.csv")



Alpha weights: [0.67360555 0.85653686 2.87389695]

Stage 1: Training with focal loss...

Epoch 1/25
158/158 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.3501 - loss: 1.1435 - val_accuracy: 0.3776 - val_loss: 0.5800 - learning_rate: 1.0000e-05
Epoch 2/25
158/158 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.3715 - loss: 1.0536 - val_accuracy: 0.4081 - val_loss: 0.6432 - learning_rate: 1.0000e-05
Epoch 3/25
158/158 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.3951 - loss: 0.9716 - val_accuracy: 0.4319 - val_loss: 0.6270 - learning_rate: 1.0000e-05
Epoch 4/25
140/158 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4002 - loss: 0.9352
Epoch 4: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-06.
158/158 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.4073 - loss: 0.9018 - val_accuracy: 0.4580 - val_loss: 0.5909 - learning_rate: 1.0000e-05
Epoch 5/25
158/158 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4112 - loss: 0.8868 - val_accuracy: 0.4663 - val_loss: 0.5791 - learn